# Lesson 09 · Representables & the Yoneda Lemma

The course’s slogan — an object is known by its arrows — becomes a theorem. And every universal property of the last three lessons turns out to be the same statement about the same kind of functor.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-09-representables-and-yoneda.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §6 — Yoneda by hand

**`yoneda_poset.py`** — an element is its down-set

In [1]:
objs = {1, 2, 3, 6}
downset = {a: {x for x in objs if a % x == 0} for a in objs}

for a in objs:
    for b in objs:
        assert (b % a == 0) == (downset[a] <= downset[b])  # a|b iff ds(a) ⊆ ds(b)

for a in sorted(objs):
    print(f"downset({a}) = {sorted(downset[a])}")

downset(1) = [1]
downset(2) = [1, 2]
downset(3) = [1, 3]
downset(6) = [1, 2, 3, 6]


**`yoneda_count.py`** — Nat(y∗, y∗) has exactly |Hom| elements

In [2]:
from itertools import product as cartesian

H = [0, 1, 2]
comp = lambda a, b: (a + b) % 3            # a . b  in Z3

nats = []
for vals in cartesian(H, repeat=3):
    alpha = dict(zip(H, vals))
    if all(alpha[comp(f, g)] == comp(alpha[f], g)   # naturality
           for f in H for g in H):
        nats.append(alpha)

print(f"natural transformations y* => y* : {len(nats)}")
for a in nats:
    assert all(a[f] == comp(a[0], f) for f in H)    # = postcompose by a[0]
    print(f"  {a}  =  postcompose by {a[0]}")

natural transformations y* => y* : 3
  {0: 0, 1: 1, 2: 2}  =  postcompose by 0
  {0: 1, 1: 2, 2: 0}  =  postcompose by 1
  {0: 2, 1: 0, 2: 1}  =  postcompose by 2


**`presheaves.py`** — presheaves on a finite category, and maps between them

In [3]:
from itertools import product as cartesian

# the category  V ==s,t==> E ;  ar[name] = (source, target)
objs = ['V', 'E']
ar   = {'iV': ('V', 'V'), 'iE': ('E', 'E'),
        's':  ('V', 'E'), 't':  ('V', 'E')}
comp = {('iV', 'iV'): 'iV', ('iE', 'iE'): 'iE', ('s', 'iV'): 's',
        ('t', 'iV'): 't', ('iE', 's'): 's', ('iE', 't'): 't'}

def hom(X, A):
    "the arrows X -> A"
    return [h for h, (d, c) in ar.items() if d == X and c == A]

class Presheaf:
    "sets[X] is F(X);  act[g] is F(g) : F(Y) -> F(X)  for  g : X -> Y"
    def __init__(self, sets, act):
        self.sets, self.act = sets, act

def y(A):
    "the representable presheaf: arrows into A, acting by precomposition"
    sets = {X: hom(X, A) for X in objs}
    act  = {g: {h: comp[(h, g)] for h in sets[c]}
            for g, (d, c) in ar.items()}
    return Presheaf(sets, act)

def nat(F, G):
    "all natural transformations F => G, by exhaustive search"
    def functions(X):
        dom, cod = F.sets[X], G.sets[X]
        return [dict(zip(dom, vals))
                for vals in cartesian(cod, repeat=len(dom))]
    out = []
    for fam in cartesian(*[functions(X) for X in objs]):
        alpha = dict(zip(objs, fam))
        if all(alpha[d][F.act[g][e]] == G.act[g][alpha[c][e]]
               for g, (d, c) in ar.items() for e in F.sets[c]):
            out.append(alpha)
    return out

print("yV:", y('V').sets, "  yE:", y('E').sets)

yV: {'V': ['iV'], 'E': []}   yE: {'V': ['s', 't'], 'E': ['iE']}


**`yoneda_graphs.py`** — the lemma and the embedding, counted

In [4]:
src = {'a': 'p', 'b': 'q', 'c': 'q', 'd': 'p'}
tgt = {'a': 'q', 'b': 'r', 'c': 'q', 'd': 'r'}
G = Presheaf({'V': ['p', 'q', 'r'], 'E': ['a', 'b', 'c', 'd']},
             {'iV': {v: v for v in 'pqr'}, 'iE': {e: e for e in 'abcd'},
              's': src, 't': tgt})

for A in objs:
    print(f"Nat(y{A}, G): {len(nat(y(A), G))}    G({A}): {len(G.sets[A])}")
for A in objs:
    for B in objs:
        print(f"Nat(y{A}, y{B}): {len(nat(y(A), y(B)))}    "
              f"arrows {A} -> {B}: {hom(A, B)}")

Nat(yV, G): 3    G(V): 3
Nat(yE, G): 4    G(E): 4
Nat(yV, yV): 1    arrows V -> V: ['iV']
Nat(yV, yE): 2    arrows V -> E: ['s', 't']
Nat(yE, yV): 0    arrows E -> V: []
Nat(yE, yE): 1    arrows E -> E: ['iE']


## Exercise 15

**`orbits.py`** — equivariant maps, both ways

In [5]:
from itertools import product as cartesian

Z3, S = range(3), range(6)
act = lambda s, g: (s + 2 * g) % 6          # the action of Z/3 on S

into_S = [a for a in cartesian(S, repeat=3)             # a[f] = alpha(f)
          if all(a[(f + g) % 3] == act(a[f], g)
                 for f in Z3 for g in Z3)]
out_of_S = [b for b in cartesian(Z3, repeat=6)          # b[s] = beta(s)
            if all(b[act(s, g)] == (b[s] + g) % 3
                   for s in S for g in Z3)]
print("y* => S :", len(into_S), "of", 6 ** 3)
print("S => y* :", len(out_of_S), "of", 3 ** 6)

y* => S : 6 of 216
S => y* : 9 of 729
